<a href="https://colab.research.google.com/github/Eswa2020/urban-parking-prediction-models/blob/master/DUBAI_Parking_project_thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Loading the datafiles

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive"  # adjust if needed
print(os.listdir(base_path))

In [ ]:
geo_path = "/content/drive/MyDrive/gcc"
print(os.listdir(geo_path))

In [ ]:
import geopandas as gpd

roads = gpd.read_file(f"{geo_path}/gis_osm_roads_free_1.shp")
pois = gpd.read_file(f"{geo_path}/gis_osm_pois_free_1.shp")
rail = gpd.read_file(f"{geo_path}/gis_osm_railways_free_1.shp")
admin = gpd.read_file(f"{geo_path}/gis_osm_places_a_free_1.shp")
landuse = gpd.read_file(f"{geo_path}/gis_osm_landuse_a_free_1.shp")
pofw=gpd.read_file((f"{geo_path}/gis_osm_pofw_a_free_1.shp"))

In [ ]:
!pip install fiona

In [ ]:
import geopandas as gpd

capacity = gpd.read_file(f"{geo_path}/Parking_RTA.kml", driver="KML")

print(capacity.head())
print(capacity.crs)
print(capacity.geometry.type.unique())

In [ ]:
print(capacity.columns)

In [ ]:
geo_path = "/content/drive/MyDrive/gcc"
print(os.listdir(geo_path))

In [ ]:
print(admin.columns)
print(admin["name"].unique())

In [ ]:
dubai = admin[
    admin["name"].str.contains("Dubai", case=False, na=False)
]

In [ ]:
print(dubai)

In [ ]:
print(dubai)
print(dubai.geometry.iloc[0])
print(dubai.area)

In [ ]:
#admin shp dont contain whole emirate we use another GADM file

In [ ]:
adm1=gpd.read_file(f"{geo_path}/ARE_adm1.shp")
print(adm1.columns)


In [ ]:
print(adm1["NAME_1"])

In [ ]:
#That contains emirates (Dubai, Abu Dhabi, Sharjah, etc.)as shown above

In [ ]:
dubai = adm1[adm1["NAME_1"] == "Dubai"]

In [ ]:
dubai.plot()

In [ ]:
#we will illustrate this using QGIS for clear mapping

In [ ]:
print(dubai.geometry.iloc[0].geom_type)

In [ ]:
#confirmation that we are dealing with a polygon

In [ ]:
roads = roads.to_crs(dubai.crs)
pois = pois.to_crs(dubai.crs)
rail = rail.to_crs(dubai.crs)
pofw=pofw.to_crs(dubai.crs)
landuse=landuse.to_crs(dubai.crs)

In [ ]:
roads_dubai = gpd.clip(roads, dubai)
pois_dubai = gpd.clip(pois, dubai)
rail_dubai = gpd.clip(rail, dubai)
pofw_dubai=pofw.to_crs(dubai.crs)
landuse_dubai = gpd.clip(landuse, dubai)
print(len(roads_dubai))
print(len(pois_dubai))
print(len(rail_dubai))
print(len(pofw_dubai))
print(len(landuse_dubai))

In [ ]:
#roads_dubai → 165,089
#pois_dubai → 14,194
#rail_dubai → 497
#pofw → 10,926
#resindential(to clip)→ 8588
#That confirms:#✔ Correct Dubai boundary#✔ Proper clipping#✔ Full spatial coverage

###Feature Engineering

In [ ]:
dubai = dubai.to_crs("EPSG:32640")
roads_dubai = roads_dubai.to_crs("EPSG:32640")
pois_dubai = pois_dubai.to_crs("EPSG:32640")
rail_dubai = rail_dubai.to_crs("EPSG:32640")
pofw_dubai=pofw_dubai.to_crs("EPSG:32640")
landuse = landuse.to_crs("EPSG:32640")

In [ ]:
capacity = capacity.to_crs("EPSG:32640")

In [ ]:
print(len(capacity))

In [ ]:
#Metro Feature (Use Stations, Not Rail Lines)
#Check rail types

In [ ]:
print(rail_dubai["fclass"].unique())

In [ ]:
#What Represents Dubai Metro?
#In Dubai:a)Metro = mostly tagged as subway,b)Dubai Tram = tagged as tram & c)Monorail (Palm) = tagged as monorail
#Heavy rail (rail, narrow_gauge) = intercity / freight → irrelevant.--->we exclude this

In [ ]:
#metro network
metro = rail_dubai[
    rail_dubai["fclass"].isin(["subway", "tram", "monorail"])
].copy()

print(len(metro))

In [ ]:
#we then compute distance to the metro line
#Fixed-rail accessibility was operationalized as Euclidean distance to metro, tram, and monorail corridors.

In [ ]:
import numpy as np
from shapely.geometry import box

# Get the bounding box of the Dubai GeoDataFrame
minx, miny, maxx, maxy = dubai.total_bounds

# Define the grid size
grid_size = 250  # meters, as used previously

# Create a grid of polygons
x_coords = np.arange(minx, maxx, grid_size)
y_coords = np.arange(miny, maxy, grid_size)

polygons = []
for x in x_coords:
    for y in y_coords:
        polygons.append(box(x, y, x + grid_size, y + grid_size))

# Create a GeoDataFrame from the grid
full_grid = gpd.GeoDataFrame(
    {"geometry": polygons},
    crs=dubai.crs # Start with the same CRS as dubai
)

# Clip the grid to the Dubai boundary and reproject to EPSG:32640
grid = gpd.clip(full_grid, dubai).to_crs("EPSG:32640")

# Add a 'grid_id' to the newly created grid
grid['grid_id'] = grid.index

In [ ]:
grid["centroid"] = grid.geometry.centroid

def min_distance(point, lines):
    return lines.distance(point).min()

grid["metro_dist"] = grid["centroid"].apply(
    lambda x: min_distance(x, metro.geometry)
)

In [ ]:
#metro_dist in meters.

In [ ]:
print(grid["metro_dist"].describe())

In [ ]:
#interpretation:
#Min ≈ 0 m → some cells lie directly on metro corridor.
#Median ≈ 38.8 km → most of Dubai is far from rail.
#Max ≈ 105 km → outer desert fringe cells.
#Mean ≈ 41 km → confirms metro network is spatially limited relative to full emirate.
#This is correct behavior.

###Compute Road Density

In [ ]:
#Intersect with your 250m grid:
grid['grid_id'] = grid.index
roads_intersect = gpd.overlay(grid, roads_dubai, how="intersection", keep_geom_type=False)
roads_intersect["length_m"] = roads_intersect.geometry.length

In [ ]:
road_sum = roads_intersect.groupby("grid_id")["length_m"].sum()

In [ ]:
grid["road_density"] = road_sum / 62500
grid["road_density"] = grid["road_density"].fillna(0)

###Compute POI density

In [ ]:
print(pois_dubai["fclass"].unique())

In [ ]:
#commercial POIs
#these generate high parking turnover
commercial_classes = [
    "mall", "department_store", "supermarket", "convenience",
    "market_place", "restaurant", "cafe", "fast_food",
    "bar", "pub", "cinema", "theatre", "hotel",
    "guesthouse", "motel", "food_court", "car_dealership"
]


In [ ]:
#institutional POIs
#generate structured parking demand
institutional_classes = [
    "school", "college", "university", "hospital",
    "clinic", "doctors", "pharmacy", "dentist",
    "police", "courthouse", "fire_station",
    "town_hall", "embassy", "community_centre"
]

In [ ]:
#now separate the 3 layers
commercial_poi = pois_dubai[
    pois_dubai["fclass"].isin(commercial_classes)
].copy()

institutional_poi = pois_dubai[
    pois_dubai["fclass"].isin(institutional_classes)
].copy()

religious_poi = pofw_dubai.copy()

In [ ]:
#compute separate densities
institutional_join = gpd.sjoin(grid, institutional_poi, predicate="intersects", how="left")

institutional_count = (
    institutional_join[~institutional_join["index_right"].isna()]
    .groupby(institutional_join[~institutional_join["index_right"].isna()].index)
    .size()
)
grid["institutional_density"] = institutional_count.fillna(0) / 62500

In [ ]:
#compute separate densities
religious_join = gpd.sjoin(grid, religious_poi, predicate="intersects", how="left")

religious_count = (
    religious_join[~religious_join["index_right"].isna()]
    .groupby(religious_join[~religious_join["index_right"].isna()].index)
    .size()
)
grid["religious_density"] = religious_count.fillna(0) / 62500

In [ ]:
#compute separate densities
commercial_join = gpd.sjoin(grid, commercial_poi, predicate="intersects", how="left")

commercial_count = (
    commercial_join[~commercial_join["index_right"].isna()]
    .groupby(commercial_join[~commercial_join["index_right"].isna()].index)
    .size()
)

grid["commercial_density"] = commercial_count.fillna(0) / 62500

### Resindential Ratio

In [ ]:
#compute Resindential Ratio
#check if their is resindential clas
residential = landuse_dubai[
    landuse_dubai["fclass"] == "residential"
].copy()

In [ ]:
print(len(residential))

In [ ]:
print(residential.head())

In [ ]:
#have to reproject the CRS first
residential = residential.to_crs(epsg=32640)

In [ ]:
res_intersect = gpd.overlay(grid, residential, how="intersection")
res_intersect["res_area"] = res_intersect.geometry.area

res_sum = res_intersect.groupby("grid_id")["res_area"].sum()

grid["res_area_sum"] = res_sum.fillna(0)
grid["res_ratio"] = grid["res_area_sum"] / 62500

###Capacity Density(RTA data)

In [ ]:
capacity_intersect = gpd.overlay(grid, capacity, how="intersection")
capacity_intersect["cap_area"] = capacity_intersect.geometry.area

cap_sum = capacity_intersect.groupby("grid_id")["cap_area"].sum()

grid["capacity_density"] = cap_sum.fillna(0) / 62500

In [ ]:
print(grid["capacity_density"].describe())

In [ ]:
print(len(grid))

In [ ]:
#If grid length > 1,148, then capacity_density is only filled for intersecting cells and others are NaN.
#Because absence of parking = zero supply, not missing.

In [ ]:
grid["capacity_density"] = grid["capacity_density"].fillna(0)

###compute imbalance (delta).

In [ ]:
grid["commercial_density"] = grid["commercial_density"].fillna(0)
grid["institutional_density"] = grid["institutional_density"].fillna(0)
grid["religious_density"] = grid["religious_density"].fillna(0)

In [ ]:
#since our POIs are structured, we define demand clearly.
grid["demand_proxy"] = (
    grid["commercial_density"] +
    grid["institutional_density"] +
    grid["religious_density"]
)

In [ ]:
grid["delta"] = grid["demand_proxy"] - grid["capacity_density"]

In [ ]:
#we clip for robustness
grid["delta_clipped"] = grid["delta"].clip(-5, 5)

In [ ]:
#confirm distribution
print(grid["delta_clipped"].describe())

In [ ]:
#Interpretation:a)Positive → demand exceeds supply,b)Negative → oversupply,c)Near zero → equilibrium
#above structurally correct.--->152,322 cells(Delta computed on full grid)--->Mean ≈ 0---->Median = 0--->Heavy concentration at zero
#makes sense because:
#Most cells have no parking and low activity
#Only limited urban cores generate imbalance
#Signal is sparse and spatially clustered
#Now we proceed to spatial structure testing.

###Moran’s I (Global Spatial Autocorrelation)

In [ ]:
import libpysal
from esda.moran import Moran

w = libpysal.weights.Queen.from_dataframe(grid)
w.transform = "r"

moran = Moran(grid["delta_clipped"], w)

print("Moran’s I:", moran.I)
print("p-value:", moran.p_sim)

In [ ]:
#Interpretation guide:
#I > 0 and p < 0.01 → clustered imbalance
#I ≈ 0 → random
#I < 0 → dispersed
#Given Dubai’s structure, positive clustering was expected
#This is strong positive spatial clustering.Shows that the imbalance is not random.It is structurally organized in space.

###Local Hotspots (Gi*)

In [ ]:
from esda.getisord import G_Local

g_local = G_Local(grid["delta_clipped"], w)

grid["GiZ"] = g_local.Zs

In [ ]:
#define statistically significant hotspots(we will use 95% threshold)
grid["giz_binary"] = 0
grid.loc[grid["GiZ"] > 1.96, "giz_binary"] = 1

In [ ]:
#then check for distribution
grid["giz_binary"] = 0
grid.loc[grid["GiZ"] > 1.96, "giz_binary"] = 1

In [ ]:
import matplotlib.pyplot as plt

grid.plot(column="GiZ", cmap="coolwarm", legend=True)
plt.title("Getis-Ord Gi* Z-Scores")
plt.show()

In [ ]:
#Interpretation:
#Red → significant high imbalance (hotspots)
#Blue → oversupply clusters
#White → no structure

In [ ]:
#was expecting the  small minority 5-15% as hotspots

In [ ]:
#Statistically Significant Hotspots Only
#we define the classes
grid["hotspot_class"] = "Not Significant"

grid.loc[grid["GiZ"] > 1.96, "hotspot_class"] = "Hotspot"
grid.loc[grid["GiZ"] < -1.96, "hotspot_class"] = "Coldspot"

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

grid.plot(
    column="hotspot_class",
    categorical=True,
    legend=True,
    cmap="coolwarm",
    linewidth=0,
    ax=ax
)

ax.set_title("Significant Parking Imbalance Clusters (Gi*)")
ax.set_axis_off()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

grid.plot(
    column="hotspot_class",
    categorical=True,
    legend=True,
    cmap="coolwarm",
    linewidth=0,
    ax=ax
)

dubai.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Significant Parking Imbalance Clusters (Gi*)")
ax.set_axis_off()
plt.show()

In [ ]:
#hotspots appear aligned to the northern corridor
#these is logical...aligns with deira,business bay and marina
#we will pull sentinel images to prove this
#The desert interior correctly shows no significance.
#Spatial structure is valid.

### Define our depedent and Independent variables

In [ ]:
#Target variable
y = grid["giz_binary"]

In [ ]:
#Predictors
features = [
    "commercial_density",
    "institutional_density",
    "religious_density",
    "road_density",
    "metro_dist",
    "res_ratio"
]
X = grid[features]

In [ ]:
# --------------------------------------------------
# Multicollinearity Diagnostics
# --------------------------------------------------

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Subset predictor variables
df_model = grid[features].dropna()

# Compute Pearson correlation matrix
corr_matrix = df_model.corr(method="pearson")

# Plot heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
    square=True
)

plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

# Identify highly correlated pairs
threshold = 0.85
high_corr_pairs = np.where(np.abs(corr_matrix) > threshold)

for i, j in zip(*high_corr_pairs):
    if i < j:
        print(
            f"High correlation between "
            f"{corr_matrix.index[i]} and {corr_matrix.columns[j]}: "
            f"{corr_matrix.iloc[i, j]:.2f}"
        )

### Split to train/Test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

### Run Random Forest classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

rf_clf.fit(X_train, y_train)

###Evaluation of model

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = rf_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
#What The Results Actually Mean
#Accuracy: 0.9615---->Looks high.---->But accuracy is misleading in imbalanced datasets.
#we have:--->Class 0 (Not hotspot): 45,338---->Class 1 (Hotspot): 359
#Hotspots are <1% of dataset.
#nb:So accuracy is not the main metric.

In [ ]:
#mportant to note:--->For Hotspots (class 1):a)Recall = 90% → Excellent means The model detects almost all true hotspots.
#& b)Precision = 16% → Weak means Many predicted hotspots are false positives.
#Precision: 0.16
#Recall: 0.90
#F1-score: 0.27

In [ ]:
#The model is highly sensitive but not highly precise.
#Meaning:a)It successfully captures most structurally significant imbalance clusters.
#b)It over-predicts hotspot membership in some areas.
#C)This is expected in rare-event spatial classification problems.

In [ ]:
#In planning contexts:THE recall of 0.90 is very strong.
#all in all its better to ,Over-identify potential hotspots (false positives)
#rather than miss real imbalance zones (false negatives).

#Improving the Random Forest classifier-Adjusting probability

In [ ]:
#Adjust probability threshold
#this usually helps in increasing precison and reducing recall
probs = rf_clf.predict_proba(X_test)[:,1]

In [ ]:
#Threshold=0.7
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
threshold = 0.7
y_pred_adj = (probs > threshold).astype(int)

print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_test, y_pred_adj))
print(classification_report(y_test, y_pred_adj))

In [ ]:
#threshold=0.8
threshold = 0.8
y_pred_adj = (probs > threshold).astype(int)

print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_test, y_pred_adj))
print(classification_report(y_test, y_pred_adj))

#Strengthen Random Forest Classifier (More Trees + Depth)

In [ ]:
#we increase stability
rf_clf_strong = RandomForestClassifier(
    n_estimators=600,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_clf_strong.fit(X_train, y_train)

probs_strong = rf_clf_strong.predict_proba(X_test)[:,1]

y_pred_strong = (probs_strong > 0.7).astype(int)

print("Stronger Model - Threshold 0.7")
print("Accuracy:", accuracy_score(y_test, y_pred_strong))
print(classification_report(y_test, y_pred_strong))

In [ ]:
#Strong RF (600 trees, depth=20, threshold=0.7)
#Hotspot:--->Precision = 0.67---->Recall = 0.48---->F1 = 0.56
#This is dramatically better.
#We keep this

In [ ]:
#Interpretation:
#The strengthened model achieved:
# Overall accuracy of 99.4%
# Hotspot precision of 0.67
# Hotspot recall of 0.48
# F1-score of 0.56

#Compared to the baseline model, this configuration significantly reduced false positives while maintaining reasonable sensitivity to spatial imbalance clusters.

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

probs = rf_clf_strong.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

print("AUC:", roc_auc)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_strong)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.show()

#Feature Importance

In [ ]:
import pandas as pd

importances = pd.Series(
    rf_clf_strong.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importances)

In [ ]:
# hotspot prediction is dominated by:
#Metro accessibility
#Road network intensity
#Not POI densities.
#meaning:
#Hotspots are structurally accessibility-driven, not just activity-driven.
#In Dubai:
#a)Imbalance clusters occur near high-accessibility corridors.
#b)Rail + arterial networks are major structural determinants.

In [ ]:
#Religious_density = 0.004
#Institutional_density = 0.013(They contribute little.)
#Commercial_density contributes more but still secondary.
#Metro_dist alone explains ~50% of predictive signal.

In [ ]:
#CONCLUSION:The Random Forest classifier revealed that metro accessibility (distance to fixed-rail infrastructure) was the strongest predictor of statistically significant imbalance clusters, followed by road density. This suggests that structural transport accessibility plays a greater role in shaping parking imbalance than localized land-use composition alone.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Extract feature importances
importances = rf_clf_strong.feature_importances_
feature_names = features  # your defined feature list

# Create DataFrame
feat_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

# Sort descending
feat_imp_df = feat_imp_df.sort_values(by="Importance", ascending=True)

# Plot
plt.figure(figsize=(10, 6))
plt.gca().set_facecolor("#e8f5e9")  # soft green background

bars = plt.barh(
    feat_imp_df["Feature"],
    feat_imp_df["Importance"],
    color="#2e7d32"
)

plt.xlabel("Relative Importance Score", fontsize=12)
plt.title("Feature Importance Ranking – Strengthened Random Forest Model",
          fontsize=14, fontweight="bold")

plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()

plt.savefig("feature_importance_final.png", dpi=300, bbox_inches="tight")
plt.show()

#Apply Model Seattle(HARTFORD)


In [ ]:
import geopandas as gpd

capacity2= gpd.read_file(f"{geo_path}/Parking_RTA.kml", driver="KML")


In [ ]:
print(capacity2.head())
print(capacity2.geometry.type.unique())

In [ ]:
#Extract the files from drive
roads_kml_path = "/content/drive/MyDrive/gcc/kenya/Roads.kml"
policest_kml_path = "/content/drive/MyDrive/gcc/kenya/Police_Stations.kml"
parks_kml_path = "/content/drive/MyDrive/gcc/kenya/Parks.kml"
hubs_kml_path = "/content/drive/MyDrive/gcc/kenya/Recreation.kml"
railw_path = "/content/drive/MyDrive/gcc/kenya/Railroad.kml"
residential_path="/content/drive/MyDrive/gcc/kenya/Building.kml"

In [ ]:
#road the various datasets
roads = gpd.read_file(roads_kml_path, driver="KML")
policest = gpd.read_file(policest_kml_path, driver="KML")
parks = gpd.read_file(parks_kml_path, driver="KML")
hubs = gpd.read_file(hubs_kml_path, driver="KML")
railw = gpd.read_file(railw_path, driver="KML")
residential = gpd.read_file(residential_path, driver="KML")

In [ ]:
#Check their individual CRS
print(capacity2.crs)
print(policest.crs)
print(hubs.crs)
print(parks.crs)
print(railw.crs)
print(roads.crs)
print(residential.crs)


In [ ]:
#Reproject CRS to one for meters..(Project to UTM)-32610
capacity2= capacity2.to_crs("EPSG:32610")
parks = parks.to_crs("EPSG:32610")
roads = roads.to_crs("EPSG:32610")
hubs= hubs.to_crs("EPSG:32610")
railw= railw.to_crs("EPSG:32610")
policest = policest.to_crs("EPSG:32610")
residential = residential.to_crs("EPSG:32610")

In [ ]:
#create 259m Grid for seattle
import numpy as np
from shapely.geometry import box

minx, miny, maxx, maxy = capacity2.total_bounds

grid_size = 250

x_coords = np.arange(minx, maxx, grid_size)
y_coords = np.arange(miny, maxy, grid_size)

polygons = []
for x in x_coords:
    for y in y_coords:
        polygons.append(box(x, y, x+grid_size, y+grid_size))

seattle_grid = gpd.GeoDataFrame(
    {"geometry": polygons},
    crs="EPSG:32610"
)

In [ ]:
#create our supply proxy(Capacity Density (Seattle))
parking_intersect = gpd.overlay(seattle_grid, capacity2, how="intersection")
parking_intersect["area"] = parking_intersect.geometry.area

area_sum = parking_intersect.groupby(parking_intersect.index)["area"].sum()

seattle_grid["capacity_density"] = area_sum.fillna(0) / 62500

In [ ]:
#create road density
roads_intersect = gpd.overlay(seattle_grid, roads, how="intersection")
roads_intersect["length"] = roads_intersect.geometry.length

road_sum = roads_intersect.groupby(roads_intersect.index)["length"].sum()

seattle_grid["road_density"] = road_sum.fillna(0) / 62500

In [ ]:
#create demand proxy(police station/recreation)
#Police station presence can be treated as:Institutional density proxy/Safety accessibility feature/ mitigation factor
#approach:Use distance to nearest police station, not count.

In [ ]:
#demand proxy(recreation)
recreation_join = gpd.sjoin(seattle_grid, hubs, how="left", predicate="intersects")

rec_count = recreation_join.groupby(recreation_join.index).size()

seattle_grid["recreation_density"] = rec_count.fillna(0) / 62500

In [ ]:
#make sure the centroids are projected 32610
seattle_grid["centroid"] = seattle_grid.geometry.centroid

In [ ]:
police_union = policest.geometry.union_all()
seattle_grid["police_dist"] = seattle_grid.centroid.distance(police_union)

In [ ]:
seattle_grid = seattle_grid.drop(columns=["centroid"])

In [ ]:
#we also compute distance to Rail(reps metro for dubai) same we did to police
railw_union = railw.geometry.union_all()
seattle_grid["rail_dist"] = seattle_grid.centroid.distance(railw_union)

In [ ]:
#We also get residential ratio using buildings
residential_join = gpd.sjoin(seattle_grid, residential, how="left", predicate="intersects")

res_count = residential_join.groupby(residential_join.index).size()

seattle_grid["residential_density"] = res_count.fillna(0) / 62500

In [ ]:
#we countinue and compute delta
seattle_grid["delta"] = (
    seattle_grid["recreation_density"]
    - seattle_grid["capacity_density"]
)

In [ ]:
seattle_grid["delta_clipped"] = seattle_grid["delta"].clip(-5, 5)

#Derive Moran-I

In [ ]:
#Moran-I Seattle
import libpysal
from esda.moran import Moran

w_sea = libpysal.weights.Queen.from_dataframe(seattle_grid)
w_sea.transform = "r"

moran_sea = Moran(seattle_grid["delta_clipped"], w_sea)

print("Seattle Moran’s I:", moran_sea.I)
print("p-value:", moran_sea.p_sim)

###Debugging Moran-I seattle Error

In [ ]:
print(seattle_grid["delta_clipped"].describe())
print(seattle_grid["delta_clipped"].var())

In [ ]:
print(len(w_sea.islands))

In [ ]:
#Good. If w_sea.islands = 0 and variance exists, then Moran’s I should not be NaN.we check the real cause.
#could be a  structural issue:maybe our delta_clipped likely contains extremely small floating values close to zero, and possibly constant after row-standardization effect.

In [ ]:
seattle_grid["delta_clipped"] = seattle_grid["delta_clipped"].fillna(0)

In [ ]:
import libpysal
from esda.moran import Moran

w_sea = libpysal.weights.Queen.from_dataframe(seattle_grid, use_index=False)
w_sea.transform = "r"

In [ ]:
y = seattle_grid["delta_clipped"].values

In [ ]:
moran_sea = Moran(y, w_sea)

print("Seattle Moran’s I:", moran_sea.I)
print("p-value:", moran_sea.p_sim)

In [ ]:
#Positive spatial autocorrelation
#a)Statistically significant,b)Weaker than Dubai (0.65),But still meaningful clustering
#actually I think is strong result.
#Dubai = highly structured imbalance
#Seattle = moderately structured imbalance

#Gi-HOTSPOT classification

In [ ]:
#Gi* Hotspot Classification (Seattle)
#Compute-Gi
from esda.getisord import G_Local

g_local_sea = G_Local(seattle_grid["delta"], w_sea)

seattle_grid["GiZ"] = g_local_sea.Zs

In [ ]:
#create a binary hotspot target(95% significance)
seattle_grid["giz_binary"] = (seattle_grid["GiZ"] > 1.96).astype(int)

In [ ]:
#check distribution
print(seattle_grid["giz_binary"].value_counts())

In [ ]:
#Given moderate clustering intensity in Hartford (Moran’s I = 0.355), a relaxed 90% confidence threshold (Z > 1.65) was adopted to identify emerging imbalance clusters.”

In [ ]:
seattle_grid["giz_binary"] = (seattle_grid["GiZ"] > 1.65).astype(int)
print(seattle_grid["giz_binary"].value_counts())

In [ ]:
seattle_grid["giz_binary"] = (seattle_grid["GiZ"] > 1.3).astype(int)
print(seattle_grid["giz_binary"].value_counts())

#Modelling Seattle

In [ ]:
#Target and Predictors variables
X_sea = seattle_grid[[
    "road_density",
    "recreation_density",
    "police_dist",
    "capacity_density",
    "rail_dist",
    "residential_density"
]]

y_sea = seattle_grid["giz_binary"]

In [ ]:
#Test/train split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_sea, y_sea, test_size=0.3, random_state=42, stratify=y_sea
)

In [ ]:
#train Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_sea = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_sea.fit(X_train, y_train)

y_pred = rf_sea.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import pandas as pd

importances = pd.Series(
    rf_sea.feature_importances_,
    index=X_sea.columns
).sort_values(ascending=False)

print(importances)

In [ ]:
#While direct cross-city classifier transfer was explored, feature heterogeneity and urban structural differences limited strict parameter alignment. Instead, replication analysis in Hartford demonstrates that the proposed spatial-statistical framework generalizes across distinct urban contexts.”

In [ ]:
seattle_grid["commercial_density"] = seattle_grid["road_density"]

In [ ]:
institution_join = gpd.sjoin(seattle_grid, policest, how="left")
inst_count = institution_join.groupby(institution_join.index).size()
seattle_grid["institutional_density"] = inst_count.fillna(0) / 62500

In [ ]:
seattle_grid["metro_dist"] = seattle_grid["rail_dist"]

In [ ]:
seattle_grid["religious_density"] = seattle_grid["recreation_density"]

In [ ]:
seattle_grid["res_ratio"] = seattle_grid["residential_density"]

In [ ]:
X_sea_transfer = seattle_grid[features]

In [ ]:
#Apply Dubai strong Model
probs_transfer = rf_clf_strong.predict_proba(X_sea_transfer)[:, 1]

seattle_grid["transfer_pred"] = (probs_transfer > 0.7).astype(int)

In [ ]:
seattle_grid["transfer_pred"].value_counts()

In [ ]:
seattle_grid["transfer_pred"].value_counts(normalize=True)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(probs_transfer, bins=30)
plt.title("Transfer Prediction Probabilities (Seattle)")
plt.xlabel("Predicted Probability")
plt.ylabel("Frequency")
plt.show()

In [ ]:
#While Hartford demonstrated significant positive spatial autocorrelation (Moran’s I = 0.355, p < 0.001), no statistically significant Gi* hotspots emerged at the 95% or 90% confidence levels. This suggests that parking imbalance in Hartford is spatially structured but not concentrated into extreme localized clusters, contrasting with the highly centralized clustering observed in Dubai.”

In [ ]:
print(probs_transfer.min())
print(probs_transfer.max())
print(probs_transfer.mean())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(probs_transfer, bins=30)
plt.title("Seattle Transfer Probability Distribution")
plt.show()

In [ ]:
print(seattle_grid.total_bounds)
#print(seattle_roads.total_bounds)

In [ ]:
len(X_sea_transfer)

In [ ]:
import numpy as np

non_zero_rows = np.where(X_sea_transfer.sum(axis=1) > 0)[0]
len(non_zero_rows)

In [ ]:
print(X_sea_transfer.describe())
print(X.describe())